In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
import sys
sys.path.append("..")

from src.temizle import Temizleyici
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import PredefinedSplit, cross_val_score
from sklearn.pipeline import Pipeline

In [ ]:
train = pd.read_csv("../data/processed/tr_train.csv", keep_default_na=False)
print(train.shape)
print(train.columns.tolist())
train.head()

In [ ]:
print("Belirsiz mesaj sayisi:", (train["belirsiz"] == "e").sum())

train = train[train["belirsiz"] != "e"].reset_index(drop=True)
print(train.shape)

In [ ]:
X=train["mesaj"]
y = (train["ikili"] == "spam").astype(int)

print(X.shape,y.shape)
print(y.value_counts()) # kaç tane 1, kaç tane 0 var
print(train["fold"].value_counts().sort_index()) # her fold’da kaç mesaj var, fold numarasına göre sıralı.

In [ ]:
kat=PredefinedSplit(train["fold"])
print("kat sayısı:", kat.get_n_splits())

In [ ]:
p = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=False, min_df=2)),
    ("model", XGBClassifier(random_state=55)),
])

skor = cross_val_score(p, X, y, cv=kat, scoring="f1")
print(skor)
print("ortalama:", skor.mean())

In [ ]:
p = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="word", sublinear_tf=True, lowercase=False,
                            token_pattern=r"\[[A-ZÇĞİÖŞÜ]+\]|\w+|[!%?:]")),
    ("model", XGBClassifier(random_state=55)),
])

skor = cross_val_score(p, X, y, cv=kat, scoring="f1")
print(skor)
print("ortalama:", skor.mean())

In [ ]:
from sklearn.neural_network import MLPClassifier

p = Pipeline([
    ("temizle", Temizleyici()),
    ("vek", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), lowercase=False, min_df=2)),
    ("model", MLPClassifier(hidden_layer_sizes=(64,), max_iter=300, random_state=55)),
])

skor = cross_val_score(p, X, y, cv=kat, scoring="f1")
print(skor)
print("ortalama:", skor.mean())

In [ ]:
sonuclar = pd.DataFrame([
    {"model": "MLP",     "özellik": "harf",  "f1": 0.974},
    {"model": "SVM",     "özellik": "harf",  "f1": 0.974},
    {"model": "XGBoost", "özellik": "kelime", "f1": 0.922},
    {"model": "XGBoost", "özellik": "harf",  "f1": 0.920},
])
sonuclar.sort_values("f1", ascending=False)